# Decoder-only GPT: MHA vs MQA/GQA, FlashAttention, MoE (DeepSeek-style)

**Перед запуском:**
1. Settings → Internet → **On**
2. Settings → Accelerator → **GPU T4 x2** (или P100)
3. В первой ячейке замени URL репозитория на свой

Сравниваемые варианты (все с одинаковым числом *активных* параметров):
- `baseline_mha` — классический MHA + manual attention (прошлое ДЗ)
- `mha_flash` — MHA + FlashAttention (SDPA)
- `gqa` / `gqa_flash` — Grouped-Query Attention (6 query, 2 KV головы)
- `mqa` — Multi-Query Attention (6 query, 1 KV голова)
- `moe_aux_free` — MoE c aux-loss-free балансировкой (DeepSeek-V3)
- `moe_aux_loss` — MoE с классическим aux loss (Switch Transformer)
- `modern_all` — GQA + Flash + MoE

In [ ]:
!rm -rf decoder-only
!git clone https://github.com/ВАШ_НИК/ВАШ_РЕПО.git decoder-only
!pip install -q comet_ml pynvml

Comet ML: закомментируй ячейку, если не нужен. Иначе вставь API key.

In [ ]:
import os
os.environ["COMET_API_KEY"] = "ВАШ_COMET_API_KEY"
os.environ["COMET_PROJECT_NAME"] = "decoder-only-shakespeare"

In [ ]:
%cd /kaggle/working/decoder-only
!nvidia-smi
!python smoke_test.py 2>&1 | tail -3

## Запуск всех экспериментов

8 экспериментов × 5000 шагов. На T4 это ~10-30 минут на эксперимент.
Чтобы запустить подмножество: `!python run_experiments.py baseline_mha gqa mqa`

In [ ]:
!python run_experiments.py

## Сравнительные графики

`results/plots/` — сравнение методов (loss, ppl, throughput, GPU util, MoE-метрики, скорость генерации) + summary-таблица.

In [ ]:
!python plot_metrics.py

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("results/plots/*.png")):
    print(p)
    display(Image(filename=p))

## Текст из лучших чекпоинтов

In [ ]:
for exp in ["baseline_mha", "gqa", "mqa", "moe_aux_free", "modern_all"]:
    print(f"\n===== {exp} =====")
    get_ipython().system(f"python sample.py --ckpt results/ckpt_{exp}_best.pt --prompt 'ROMEO:' --max-new-tokens 300")

Чекпоинты, метрики (.jsonl) и графики — в Output → `decoder-only/results/`.